# Governance Auto-Delete

Deletes Fabric workspace items that have passed the full 3-strike warning cycle with no owner action.

### Safety checks (in order)
1. `enable_auto_delete` must be `true` in config
2. `dry_run` must be `false` in config
3. Item must have status `deletion_ready` or `warning_3` in tracker
4. Item type must NOT be in `protected_types`
5. Item ID must NOT be in `protected_items`
6. Item must still exist in the workspace (re-verified via API)
7. Item must still be stale (active-item guard — re-checked from latest snapshot)
8. This must NOT be the first pipeline run (no bulk deletion on first run)

Only after **all 8 checks pass** is an item deleted.

### Permission required
**Contributor** or **Admin** on the target workspace (must be able to delete items).

### API used
```
DELETE https://api.fabric.microsoft.com/v1/workspaces/{workspaceId}/items/{itemId}
```


## 1. Setup

In [ ]:
import requests
import pandas as pd
import time
import uuid
import logging
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("auto_delete")

import notebookutils

FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"

pipeline_run_id = str(uuid.uuid4())
run_timestamp   = datetime.now(timezone.utc).isoformat()
run_date_str    = datetime.now(timezone.utc).strftime("%B %d, %Y")

log.info(f"Pipeline run ID: {pipeline_run_id}")
log.info(f"Run timestamp:   {run_timestamp}")

# Authentication
token = notebookutils.credentials.getToken("pbi")
token_acquired_at = time.time()

def get_headers():
    global token, token_acquired_at
    if time.time() - token_acquired_at > 2400:
        log.info("Refreshing token...")
        token = notebookutils.credentials.getToken("pbi")
        token_acquired_at = time.time()
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

log.info("Authentication successful.")


## 2. Load configuration and data

In [ ]:
# Config
df_cfg = spark.sql("SELECT config_key, config_value FROM governance_config").toPandas()
config = dict(zip(df_cfg["config_key"], df_cfg["config_value"]))

enable_auto_delete = config.get("enable_auto_delete", "false").lower() == "true"
dry_run            = config.get("dry_run", "true").lower() == "true"
protected_types    = [t.strip() for t in config.get("protected_types", "").split(",") if t.strip()]
protected_items    = [i.strip() for i in config.get("protected_items", "").split(",") if i.strip()]
admin_email        = config.get("admin_email", "SURYADEV.RATHORE@XEBIA.COM")
workspace_ids      = [w.strip() for w in config.get("workspace_ids", "").split(",") if w.strip()]

log.info(f"enable_auto_delete: {enable_auto_delete}")
log.info(f"dry_run:            {dry_run}")
log.info(f"protected_types:    {protected_types}")
log.info(f"protected_items:    {len(protected_items)} items")

# Tracker — items ready for deletion
df_tracker = spark.sql("SELECT * FROM cleanup_tracker").toPandas()
for col in ["warning_count", "cleanup_score"]:
    if col in df_tracker.columns:
        df_tracker[col] = pd.to_numeric(df_tracker[col], errors="coerce")

deletion_statuses = {"deletion_ready", "pending_deletion"}
candidates = df_tracker[df_tracker["status"].isin(deletion_statuses)].copy() if not df_tracker.empty else pd.DataFrame()

log.info(f"Tracker total: {len(df_tracker)} items")
log.info(f"Deletion candidates: {len(candidates)} items")

# Latest snapshot for active-item guard re-check
df_snap = spark.sql("""
    SELECT id, is_stale, days_since_modified, days_since_last_used, last_used_date
    FROM workspace_inventory_snapshot
    WHERE snapshot_id = (
        SELECT snapshot_id FROM workspace_inventory_snapshot
        ORDER BY snapshot_time_utc DESC LIMIT 1
    )
""").toPandas()
for col in ["is_stale", "days_since_modified", "days_since_last_used"]:
    if col in df_snap.columns:
        df_snap[col] = pd.to_numeric(df_snap[col], errors="coerce")

log.info(f"Snapshot for re-check: {len(df_snap)} items")

# Check if this is the first run (audit log has entries from previous runs)
df_prev_runs = spark.sql("SELECT DISTINCT pipeline_run_id FROM cleanup_audit_log").toPandas()
is_first_run = len(df_prev_runs) <= 1  # 0 or 1 means this is first/second run
log.info(f"Previous pipeline runs: {len(df_prev_runs)} (first_run_guard: {is_first_run})")


## 3. Gate checks

In [ ]:
# ── Gate 1: Master switch ──────────────────────────────
if not enable_auto_delete:
    log.warning("GATE 1 BLOCKED: enable_auto_delete = false")
    log.info("Set governance_config: enable_auto_delete = 'true' to enable.")
    log.info("No items will be deleted. Exiting.")
    candidates = pd.DataFrame()  # Empty = skip all

# ── Gate 2: Dry run ───────────────────────────────────
elif dry_run:
    log.warning("GATE 2: dry_run = true — will simulate deletions only")
    log.info("Set governance_config: dry_run = 'false' for actual deletion.")

# ── Gate 3: First run guard ───────────────────────────
elif is_first_run:
    log.warning("GATE 3 BLOCKED: First/second pipeline run — no deletions allowed")
    log.info("Run the pipeline at least 2 more times before deletion is permitted.")
    candidates = pd.DataFrame()

# ── Gate 4: No candidates ────────────────────────────
elif candidates.empty:
    log.info("GATE 4: No items in deletion_ready status. Nothing to delete.")

else:
    log.info(f"All gates passed. {len(candidates)} items eligible for deletion review.")


## 4. Pre-deletion safety checks per item

In [ ]:
items_to_delete = []
items_skipped = []

for _, item in candidates.iterrows():
    item_id   = str(item.get("item_id", ""))
    item_name = str(item.get("item_name", ""))
    item_type = str(item.get("item_type", ""))
    owner     = str(item.get("owner_email", ""))
    ws_id     = str(item.get("workspace_id", ""))
    score     = item.get("cleanup_score", 0)

    skip_reason = None

    # Check 1: Protected type
    if item_type in protected_types:
        skip_reason = f"Protected type: {item_type}"

    # Check 2: Protected item ID
    elif item_id in protected_items:
        skip_reason = "Protected item (exempted by ID)"

    # Check 3: Active-item guard — re-check from latest snapshot
    elif not df_snap.empty:
        snap_row = df_snap[df_snap["id"].astype(str) == item_id]
        if not snap_row.empty:
            s = snap_row.iloc[0]

            # Recently modified?
            dsm = s.get("days_since_modified")
            if pd.notna(dsm) and float(dsm) < 14:
                skip_reason = f"Active-item guard: modified {int(dsm)} days ago"

            # Recently used?
            dsu = s.get("days_since_last_used")
            if not skip_reason and pd.notna(dsu) and float(dsu) < 30:
                skip_reason = f"Active-item guard: used {int(dsu)} days ago"

            # No longer stale?
            is_stale = s.get("is_stale")
            if not skip_reason and pd.notna(is_stale) and int(is_stale) == 0:
                skip_reason = "Active-item guard: item is no longer stale"

    # Check 4: Item must still exist (verified during actual deletion)
    # (Checked in the deletion step itself)

    if skip_reason:
        items_skipped.append({
            "item_id": item_id, "item_name": item_name,
            "item_type": item_type, "owner": owner, "reason": skip_reason
        })
        log.info(f"  SKIP: {item_name} ({item_type}) — {skip_reason}")
    else:
        items_to_delete.append({
            "item_id": item_id, "item_name": item_name,
            "item_type": item_type, "owner_email": owner,
            "workspace_id": ws_id, "cleanup_score": score,
            "warning_1_date": item.get("warning_1_date", ""),
            "warning_2_date": item.get("warning_2_date", ""),
            "warning_3_date": item.get("warning_3_date", ""),
        })

log.info(f"")
log.info(f"Pre-deletion check complete:")
log.info(f"  Ready to delete:  {len(items_to_delete)}")
log.info(f"  Skipped (safety): {len(items_skipped)}")


## 5. Execute deletions

In [ ]:
deleted_items = []
failed_items = []

def delete_fabric_item(workspace_id, item_id, max_retries=3):
    """Delete a single item via Fabric REST API. Returns (success, status_code, message)."""
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items/{item_id}"
    
    for attempt in range(max_retries):
        resp = requests.delete(url, headers=get_headers())
        
        if resp.status_code in (200, 204):
            return True, resp.status_code, "Deleted successfully"
        elif resp.status_code == 404:
            return True, 404, "Item already gone (deleted by owner)"
        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 10))
            log.warning(f"  Throttled. Waiting {wait}s (attempt {attempt+1})...")
            time.sleep(wait)
        elif resp.status_code in (401, 403):
            return False, resp.status_code, f"Permission denied: {resp.text[:200]}"
        else:
            return False, resp.status_code, f"Unexpected: {resp.text[:200]}"
    
    return False, 429, "Failed after max retries (throttled)"

if items_to_delete:
    for i, item in enumerate(items_to_delete):
        item_id   = item["item_id"]
        item_name = item["item_name"]
        ws_id     = item["workspace_id"]

        if dry_run:
            # Simulate deletion
            log.info(f"  [DRY RUN] Would delete: {item_name} ({item['item_type']}) from workspace {ws_id}")
            deleted_items.append(item)
        else:
            # Actual deletion
            log.info(f"  Deleting [{i+1}/{len(items_to_delete)}]: {item_name} ({item['item_type']})...")
            success, status, msg = delete_fabric_item(ws_id, item_id)
            
            if success:
                log.info(f"    ✔ {msg} (HTTP {status})")
                deleted_items.append(item)
            else:
                log.error(f"    ✗ {msg} (HTTP {status})")
                failed_items.append({**item, "error": msg, "status_code": status})
            
            # Rate-limit courtesy: 1 second between deletions
            if i < len(items_to_delete) - 1:
                time.sleep(1)

    log.info(f"")
    log.info(f"Deletion complete:")
    log.info(f"  {'Simulated' if dry_run else 'Deleted'}:  {len(deleted_items)}")
    log.info(f"  Failed:   {len(failed_items)}")
else:
    log.info("No items to delete.")


## 6. Update tracker and audit log

In [ ]:
tracker_rows = df_tracker.to_dict("records") if not df_tracker.empty else []
tracker_by_id = {str(r["item_id"]): i for i, r in enumerate(tracker_rows)}
audit_entries = []
outbox_rows = []

# ── Mark deleted items ────────────────────────────────
for item in deleted_items:
    idx = tracker_by_id.get(item["item_id"])
    if idx is not None:
        tracker_rows[idx]["status"] = "deleted"
        tracker_rows[idx]["deleted_date"] = run_timestamp
        tracker_rows[idx]["last_updated"] = run_timestamp
        tracker_rows[idx]["pipeline_run_id"] = pipeline_run_id

    audit_entries.append({
        "audit_id": str(uuid.uuid4()),
        "timestamp": run_timestamp,
        "pipeline_run_id": pipeline_run_id,
        "item_id": item["item_id"],
        "item_name": item["item_name"],
        "item_type": item["item_type"],
        "owner_email": item["owner_email"],
        "action": "auto_deleted_dry_run" if dry_run else "auto_deleted",
        "detail": f"Score: {item.get('cleanup_score', 0)}, warnings: 3",
        "workspace_id": item.get("workspace_id", ""),
    })

# ── Mark skipped items back to warning_3 (re-check next run) ──
for item in items_skipped:
    idx = tracker_by_id.get(item["item_id"])
    if idx is not None:
        # If skipped due to active-item guard, resolve it
        if "Active-item guard" in item["reason"]:
            tracker_rows[idx]["status"] = "resolved"
            tracker_rows[idx]["resolved_date"] = run_timestamp
            action = "owner_resolved"
        else:
            tracker_rows[idx]["status"] = "warning_3"  # Stay at warning_3, retry next run
            action = "deletion_skipped"
        tracker_rows[idx]["last_updated"] = run_timestamp
        tracker_rows[idx]["pipeline_run_id"] = pipeline_run_id

        audit_entries.append({
            "audit_id": str(uuid.uuid4()),
            "timestamp": run_timestamp,
            "pipeline_run_id": pipeline_run_id,
            "item_id": item["item_id"],
            "item_name": item["item_name"],
            "item_type": item["item_type"],
            "owner_email": item["owner"],
            "action": action,
            "detail": f"Skipped: {item['reason']}",
            "workspace_id": "",
        })

# ── Mark failed items ─────────────────────────────────
for item in failed_items:
    audit_entries.append({
        "audit_id": str(uuid.uuid4()),
        "timestamp": run_timestamp,
        "pipeline_run_id": pipeline_run_id,
        "item_id": item["item_id"],
        "item_name": item["item_name"],
        "item_type": item["item_type"],
        "owner_email": item["owner_email"],
        "action": "deletion_failed",
        "detail": f"HTTP {item.get('status_code', '?')}: {item.get('error', 'Unknown')}",
        "workspace_id": item.get("workspace_id", ""),
    })

log.info(f"Audit entries prepared: {len(audit_entries)}")


## 7. Generate deletion confirmation emails

In [ ]:
# ── Xebia email styling (same as Email Generator) ─────
XEBIA_PURPLE = "#6a1b6a"
WHITE        = "#ffffff"
GRAY_BG      = "#f7f7f7"
GRAY_BORDER  = "#e0e0e0"
GRAY_TEXT    = "#666666"
BLACK_TEXT   = "#333333"
logo_url     = config.get("logo_url", "")

if deleted_items and not dry_run:
    # Group deleted items by owner
    from collections import defaultdict
    by_owner = defaultdict(list)
    for item in deleted_items:
        by_owner[item["owner_email"]].append(item)

    for owner, items in by_owner.items():
        if not owner or pd.isna(owner) or owner in ("", "nan", "None"):
            continue

        first_name = owner.split("@")[0].replace(".", " ").split()
        first_name = first_name[0].title() if first_name else "Team member"
        n = len(items)

        logo_html = f'<img src="{logo_url}" alt="Xebia" height="32"/>' if logo_url else '<span style="font-size:24px;font-weight:700;color:#ffffff;letter-spacing:1px;">Xebia</span>'

        # Build items table rows
        rows_html = ""
        for i, item in enumerate(items):
            bg = WHITE if i % 2 == 0 else GRAY_BG
            w1 = str(item.get("warning_1_date", "—"))[:10] if item.get("warning_1_date") else "—"
            w2 = str(item.get("warning_2_date", "—"))[:10] if item.get("warning_2_date") else "—"
            w3 = str(item.get("warning_3_date", "—"))[:10] if item.get("warning_3_date") else "—"
            rows_html += f"""<tr>
                <td style="background:{bg};padding:8px 12px;border-bottom:1px solid {GRAY_BORDER};">{item['item_name']}</td>
                <td style="background:{bg};padding:8px 12px;border-bottom:1px solid {GRAY_BORDER};">{item['item_type']}</td>
                <td style="background:{bg};padding:8px 12px;border-bottom:1px solid {GRAY_BORDER};">{int(item.get('cleanup_score', 0))}</td>
                <td style="background:{bg};padding:8px 12px;border-bottom:1px solid {GRAY_BORDER};">{w1}, {w2}, {w3}</td>
            </tr>"""

        body = f"""
        <table width="100%" cellpadding="0" cellspacing="0" style="max-width:680px;margin:0 auto;font-family:Segoe UI,Arial,sans-serif;">
        <tr><td style="background:{XEBIA_PURPLE};padding:20px 28px;border-radius:8px 8px 0 0;">
            <table width="100%" cellpadding="0" cellspacing="0">
            <tr><td>{logo_html}</td>
                <td style="text-align:right;color:rgba(255,255,255,0.8);font-size:12px;">Fabric Workspace Governance</td></tr>
            </table>
        </td></tr>
        <tr><td style="background:{WHITE};padding:24px 28px;border-left:1px solid {GRAY_BORDER};border-right:1px solid {GRAY_BORDER};">
            <h1 style="margin:0 0 4px;font-size:20px;font-weight:600;color:{BLACK_TEXT};">{n} items removed from workspace</h1>
            <p style="margin:0 0 16px;font-size:13px;color:{GRAY_TEXT};">{run_date_str}</p>
            <p style="font-size:14px;color:{BLACK_TEXT};">Hi {first_name},</p>
            <p style="font-size:14px;color:{BLACK_TEXT};">
                The following items were automatically removed from the workspace
                after 3 cleanup notices with no action taken.
            </p>
            <table width="100%" cellpadding="0" cellspacing="0" style="border:1px solid {GRAY_BORDER};border-radius:6px;border-collapse:separate;font-size:13px;">
                <tr>
                    <th style="background:#f3e8f3;color:#4a124a;padding:8px 12px;text-align:left;font-weight:600;font-size:12px;border-bottom:1px solid {GRAY_BORDER};">Name</th>
                    <th style="background:#f3e8f3;color:#4a124a;padding:8px 12px;text-align:left;font-weight:600;font-size:12px;border-bottom:1px solid {GRAY_BORDER};">Type</th>
                    <th style="background:#f3e8f3;color:#4a124a;padding:8px 12px;text-align:left;font-weight:600;font-size:12px;border-bottom:1px solid {GRAY_BORDER};">Score</th>
                    <th style="background:#f3e8f3;color:#4a124a;padding:8px 12px;text-align:left;font-weight:600;font-size:12px;border-bottom:1px solid {GRAY_BORDER};">Warnings sent</th>
                </tr>
                {rows_html}
            </table>
            <p style="margin:16px 0 0;font-size:13px;color:{GRAY_TEXT};background:{GRAY_BG};padding:10px 14px;border-radius:6px;">
                If you believe any item was removed in error, please contact the workspace administrator within 48 hours.
            </p>
        </td></tr>
        <tr><td style="background:{GRAY_BG};padding:16px 28px;border-radius:0 0 8px 8px;border:1px solid {GRAY_BORDER};border-top:none;text-align:center;">
            <p style="margin:0;font-size:12px;color:{GRAY_TEXT};">Xebia &bull; Fabric Workspace Governance &bull; Automated notification</p>
        </td></tr>
        </table>
        """

        outbox_rows.append({
            "email_id": str(uuid.uuid4()),
            "pipeline_run_id": pipeline_run_id,
            "email_type": "deletion_confirmation",
            "recipient": owner,
            "subject": f"{n} workspace items have been removed",
            "body_html": body,
            "status": "pending",
            "created_at": run_timestamp,
            "workspace_id": items[0].get("workspace_id", ""),
        })

    log.info(f"Deletion confirmation emails prepared: {len(outbox_rows)}")

elif deleted_items and dry_run:
    log.info(f"[DRY RUN] Would send {len(set(i['owner_email'] for i in deleted_items))} deletion confirmation emails")
else:
    log.info("No deletion emails needed.")


## 8. Save all changes

In [ ]:
# ── Save tracker ───────────────────────────────────────
if tracker_rows:
    df_updated = pd.DataFrame(tracker_rows)
    for col in ["warning_count", "cleanup_score"]:
        if col in df_updated.columns:
            df_updated[col] = pd.to_numeric(df_updated[col], errors="coerce").fillna(0).astype(int)
    spark.createDataFrame(df_updated.astype(str)).write.format("delta").mode("overwrite").saveAsTable("cleanup_tracker")
    log.info(f"✔ cleanup_tracker saved: {len(df_updated)} rows")

# ── Append audit log ──────────────────────────────────
if audit_entries:
    spark.createDataFrame(pd.DataFrame(audit_entries).astype(str)).write.format("delta").mode("append").saveAsTable("cleanup_audit_log")
    log.info(f"✔ cleanup_audit_log: {len(audit_entries)} entries appended")

test_recipient = str(config.get("test_mode_recipient", "")).strip()
if test_recipient and test_recipient.lower() not in ("none", "nan") and outbox_rows:
    log.warning(f"TEST MODE — {len(outbox_rows)} deletion emails redirected to {test_recipient}")
    for row in outbox_rows:
        original = row["recipient"]
        row["recipient"] = test_recipient
        row["subject"] = f"[TEST -> {original}] {row['subject']}"

# ── Append email outbox ───────────────────────────────
if outbox_rows:
    spark.createDataFrame(pd.DataFrame(outbox_rows).astype(str)).write.format("delta").mode("append").saveAsTable("email_outbox")
    log.info(f"✔ email_outbox: {len(outbox_rows)} emails appended")


## 9. Summary

In [ ]:
print("=" * 60)
print("  GOVERNANCE AUTO-DELETE — SUMMARY")
print("=" * 60)
print(f"  Pipeline run:         {pipeline_run_id}")
print(f"  Timestamp:            {run_timestamp}")
print(f"  Mode:                 {'DRY RUN (no actual deletions)' if dry_run else 'LIVE'}")
print(f"  Auto-delete enabled:  {enable_auto_delete}")
print(f"  ")
print(f"  GATE CHECKS")
print(f"  ───────────")
print(f"  enable_auto_delete:   {'PASS' if enable_auto_delete else 'BLOCKED'}")
print(f"  dry_run:              {'SIMULATING' if dry_run else 'PASS (live mode)'}")
print(f"  first_run_guard:      {'BLOCKED' if is_first_run else 'PASS'}")
print(f"  candidates found:     {len(candidates)}")
print(f"  ")
print(f"  RESULTS")
print(f"  ───────")
print(f"  Items {'simulated' if dry_run else 'deleted'}:       {len(deleted_items)}")
print(f"  Items skipped (safety): {len(items_skipped)}")
print(f"  Items failed:           {len(failed_items)}")
print(f"  ")
if items_skipped:
    print(f"  SKIPPED ITEMS")
    print(f"  ─────────────")
    for s in items_skipped:
        print(f"  {s['item_name']:30s} {s['reason']}")
    print(f"  ")
if failed_items:
    print(f"  FAILED ITEMS")
    print(f"  ────────────")
    for f_item in failed_items:
        print(f"  {f_item['item_name']:30s} HTTP {f_item.get('status_code','?')}: {f_item.get('error','')[:50]}")
    print(f"  ")
print(f"  OUTPUTS")
print(f"  ───────")
print(f"  Audit entries:          {len(audit_entries)}")
print(f"  Confirmation emails:    {len(outbox_rows)}")
print("=" * 60)


## Safety configuration reference

To enable actual deletion, update these two settings in `governance_config`:

```sql
UPDATE governance_config SET config_value = 'true' WHERE config_key = 'enable_auto_delete';
UPDATE governance_config SET config_value = 'false' WHERE config_key = 'dry_run';
```

To protect specific items from ever being deleted:

```sql
UPDATE governance_config SET config_value = 'item-guid-1,item-guid-2' WHERE config_key = 'protected_items';
```

To revert to safe mode:

```sql
UPDATE governance_config SET config_value = 'false' WHERE config_key = 'enable_auto_delete';
UPDATE governance_config SET config_value = 'true' WHERE config_key = 'dry_run';
```
